---
last_verified: 2026-08-31
tool_version: 3.2.4
sources:
  - https://httpie.io/docs/cli#status-code-checking
  - https://httpie.io/docs/cli/example-use-cases
---

# Comparing httpie --check-status vs curl -f for CI smoke-test gating

> L4 notebook — evaluating two HTTP CLIs for CI health-check gating: exit-code semantics, timeout behavior, and failure-mode coverage.

## Purpose

CI pipelines commonly run a lightweight smoke test after deployment to confirm the service is reachable and returning valid responses. Two dominant approaches exist: HTTPie's `--check-status` flag and curl's `-f` (or `--fail`) flag. Both cause the CLI to exit non-zero on server errors, but they differ in which status codes they treat as failures, how they handle timeouts, and what output they produce on failure. This notebook walks through both approaches against the same set of test scenarios and compares the results.

## When to use

Use `--check-status` when the pipeline needs granular control over which HTTP status codes count as failures — for example, treating 404 as acceptable during a canary rollout but treating 500 as a hard gate. Use `curl -f` when the goal is a simple binary check: any non-2xx response means the service is unhealthy. Both work in GitHub Actions, GitLab CI, and Jenkins without additional dependencies beyond the CLI itself.

## Prerequisites

- httpie 3.2.4 installed (`pip install httpie`).
- curl available (pre-installed on most Linux/macOS images).
- A target endpoint that returns different status codes on demand. The examples below use `https://httpbin.org` which supports status-code injection via `/status/<code>`.

## Step 1 — Baseline: both tools on a healthy endpoint

Both `http --check-status` and `curl -f` exit 0 when the server returns a 2xx status code. The difference shows up only on failure.

In [ ]:
# Both exit 0 on a 200 response
http --check-status GET https://httpbin.org/status/200
echo "httpie exit code: $?"

curl -f -s -o /dev/null https://httpbin.org/status/200
echo "curl exit code: $?"

## Step 2 — 4xx client error: where the behavior diverges

HTTPie's `--check-status` exits non-zero for any status code >= 400. Curl's `-f` also exits non-zero for 4xx and 5xx, but the default exit code is 22 (HTTP error detected), which some CI runners interpret differently from a generic non-zero exit.

In [ ]:
# 404 — both should fail
http --check-status GET https://httpbin.org/status/404
echo "httpie 404 exit code: $?"

curl -f -s -o /dev/null https://httpbin.org/status/404
echo "curl 404 exit code: $?"

## Step 3 — Selective status-code gating with httpie

HTTPie allows passing `--check-status` with a list of acceptable status codes. This is useful when the smoke test should tolerate specific error codes — for example, a 404 on a resource that may not exist yet during a canary deploy.

In [ ]:
# --check-status with acceptable codes: 404 is allowed, 500 is not
http --check-status=200,404 GET https://httpbin.org/status/404
echo "httpie 404 (allowed) exit code: $?"

http --check-status=200,404 GET https://httpbin.org/status/500
echo "httpie 500 (not allowed) exit code: $?"

Curl has no equivalent to this selective gating. To achieve the same result with curl, the pipeline must inspect the response code manually:

In [ ]:
# Manual status-code check with curl
STATUS=$(curl -s -o /dev/null -w '%{http_code}' https://httpbin.org/status/404)
if [[ "$STATUS" == "200" || "$STATUS" == "404" ]]; then
    echo "curl: status $STATUS — acceptable"
else
    echo "curl: status $STATUS — FAIL"
    exit 1
fi

## Step 4 — Timeout behavior

Both tools support connection and read timeouts, but the defaults and flag names differ. In CI, setting explicit timeouts prevents hanging jobs.

In [ ]:
# httpie: --timeout in seconds (float)
http --check-status --timeout=5 GET https://httpbin.org/delay/10 2>&1 || echo "httpie timed out (exit $?"

# curl: --connect-timeout and --max-time in seconds
curl -f --connect-timeout 3 --max-time 5 -s -o /dev/null https://httpbin.org/delay/10 2>&1 || echo "curl timed out (exit $?"

Key differences:
- HTTPie uses a single `--timeout` flag that covers the entire request lifecycle.
- Curl separates connection timeout (`--connect-timeout`) from total transfer time (`--max-time`).
- Both tools exit non-zero on timeout, which is the correct behavior for CI gating.

## Step 5 — Output on failure

In CI, the failure output determines how quickly a developer can diagnose the problem. Both tools can suppress output or redirect it, but the defaults differ.

In [ ]:
# httpie prints response headers and body by default on failure
http --check-status GET https://httpbin.org/status/500 2>&1 | head -5

# curl -f suppresses the response body on error; use -v for full output
curl -f -s -o /dev/null https://httpbin.org/status/500 2>&1; echo "curl exit: $?"

# curl -v shows full request/response on failure
curl -f -v https://httpbin.org/status/500 2>&1 | tail -10

For CI logs, httpie's default verbosity is often more useful — it shows the response body without extra flags. Curl requires `-v` for the same level of detail, but `-v` also dumps request headers which can leak secrets if the pipeline sets auth tokens.

## Step 6 — Combining both in a CI pipeline

A practical CI job often needs both tools: httpie for its selective gating and readable output, curl for its ubiquity and download support. The pattern below uses httpie as the primary gate and curl as a fallback health check.

In [ ]:
#!/bin/bash
# ci-smoke-test.sh — combined httpie + curl health gate
set -euo pipefail

BASE_URL="${1:-https://httpbin.org}"

# Primary gate: httpie with selective status-code tolerance
echo "=== httpie gate ==="
http --check-status=200,201,202 --timeout=10 \
     GET "$BASE_URL/status/200" || { echo "FAIL: httpie gate"; exit 1; }

# Secondary gate: curl simple fail-on-error
echo "=== curl gate ==="
curl -f -s -o /dev/null --max-time 10 \
     "$BASE_URL/status/200" || { echo "FAIL: curl gate"; exit 1; }

echo "All smoke tests passed."

## Comparison summary

| Aspect | httpie `--check-status` | curl `-f` |
|--------|------------------------|-----------|
| Default failure codes | >= 400 | any non-2xx |
| Selective gating | built-in (`--check-status=200,404`) | manual status-code inspection |
| Timeout flag | `--timeout` (single) | `--connect-timeout` + `--max-time` (two flags) |
| Failure output | headers + body by default | suppressed; requires `-v` |
| Exit code on error | 1 (generic non-zero) | 22 (curl-specific HTTP error code) |
| Pre-installed on CI images | no (`pip install httpie`) | yes (typically available) |
| JSON output formatting | default (colorized) | requires `jq` or `--json` flag (curl 7.80+)

## Common errors

- **curl exit code 22 vs 1:** CI scripts that check `exit 1` specifically will miss curl's HTTP error exit code of 22. Use `$? -ne 0` instead of `$? -eq 1`.
- **httpie not installed on CI image:** The `http` command is not available on default GitHub Actions runners. Either install it explicitly (`pip install httpie`) or guard the call with `command -v http`.
- **Leaking auth headers in curl -v output:** Using `-v` to debug a failed curl gate dumps request headers including `Authorization`. Pipe through `grep -v Authorization` or use `--trace-ascii` with a filter in sensitive environments.
- **Timeout units differ:** HTTPie's `--timeout` is in seconds (supports decimals). Curl's `--max-time` is also in seconds but truncates to integer on some builds.

## What to try next

Two areas worth investigating: (1) how HTTPie's `--session` interacts with `--check-status` when the session carries stale auth headers — does the status check fire before or after the session state is applied? (2) Whether combining httpie's `--check-status` with `--download` produces a reliable file-download gate for CI artifact fetching, or if curl's `-f -O` remains the simpler path.